# Task 1: Tokenization and a FAQ Chatbot

Lab 6, CAI601400. All FAQ text and examples are synthetic teaching data.

## Part 1: Tokenization

Comparing whitespace splitting (`text.split()`) with a Unicode word regex (`\w+`) on six inputs: punctuation, a contraction, an email, an identifier (CS4944), an empty string and Urdu text.

In [1]:
import re

texts = [
    "Hello, hey! Is this working?",      # punctuation
    "I don't know. I think it is.a",     # contraction + missing space after a full stop
    "Email me at help@example.com",      # email (example.com is a reserved demo domain)
    "CS4944 is my AI course",            # identifier
    "",                                  # empty string
    "میں طالب علم ہوں۔",                 # Urdu (ends with the Urdu full stop)
]


def whitespaceTokens(text):
    return text.split()


def regexTokens(text):
    return re.findall(r"\w+", text)

In [2]:
for text in texts:
    spaceList = whitespaceTokens(text)
    regexList = regexTokens(text)

    print("Original text:", repr(text))
    print("Whitespace tokens:", spaceList, "| Count:", len(spaceList))
    print("Regex tokens:", regexList, "| Count:", len(regexList))
    print()

Original text: 'Hello, hey! Is this working?'
Whitespace tokens: ['Hello,', 'hey!', 'Is', 'this', 'working?'] | Count: 5
Regex tokens: ['Hello', 'hey', 'Is', 'this', 'working'] | Count: 5

Original text: "I don't know. I think it is.a"
Whitespace tokens: ['I', "don't", 'know.', 'I', 'think', 'it', 'is.a'] | Count: 7
Regex tokens: ['I', 'don', 't', 'know', 'I', 'think', 'it', 'is', 'a'] | Count: 9

Original text: 'Email me at help@example.com'
Whitespace tokens: ['Email', 'me', 'at', 'help@example.com'] | Count: 4
Regex tokens: ['Email', 'me', 'at', 'help', 'example', 'com'] | Count: 6

Original text: 'CS4944 is my AI course'
Whitespace tokens: ['CS4944', 'is', 'my', 'AI', 'course'] | Count: 5
Regex tokens: ['CS4944', 'is', 'my', 'AI', 'course'] | Count: 5

Original text: ''
Whitespace tokens: [] | Count: 0
Regex tokens: [] | Count: 0

Original text: 'میں طالب علم ہوں۔'
Whitespace tokens: ['میں', 'طالب', 'علم', 'ہوں۔'] | Count: 4
Regex tokens: ['میں', 'طالب', 'علم', 'ہوں'] | Count: 4



### What the comparison shows

**Difference 1: punctuation.** Whitespace splitting keeps punctuation attached to words (`'Hello,'`, `'hey!'`, `'working?'`, and the Urdu `'ہوں۔'` with its full stop). The same word can therefore become different tokens depending on what follows it. The regex drops punctuation, so `'Hello'` and `'working'` are clean, but the information carried by `?` or `!` is lost.

**Difference 2: contractions, emails and missing spaces.** The regex splits `don't` into `'don'` and `'t'` (9 tokens against 7 for the second sentence), splits the email into `'help'`, `'example'`, `'com'` (6 against 4), and splits `is.a` into `'is'` and `'a'` where whitespace keeps `'is.a'` whole. Neither output is "right"; each changes token counts and what a later search would match. `CS4944` and the empty string behave the same in both methods.

**One normalization risk.** Lowercasing and stripping punctuation can destroy meaning. The cell below shows how an endpoint path and a negation are changed. The original text must be kept for display and evidence, with any normalized copy used only as a separate search aid.

**Caution.** The regex only extracts runs of letters and digits. It is not a complete multilingual tokenizer: the Urdu sentence gave the same four tokens as whitespace splitting only because its words happen to be separated by spaces, and this says nothing about Urdu morphology or scripts written without spaces.

In [3]:
for text in ["Do NOT reset the password", "POST /v2/token/refresh"]:
    normalized = " ".join(regexTokens(text.lower()))
    print("Original  :", repr(text))
    print("Normalized:", repr(normalized))
    print()

Original  : 'Do NOT reset the password'
Normalized: 'do not reset the password'

Original  : 'POST /v2/token/refresh'
Normalized: 'post v2 token refresh'



### Model tokenizer (two English examples)

The lab's encoder is `sentence-transformers/all-MiniLM-L6-v2`, so its own tokenizer is used here (not `tiktoken`, which belongs to a different model family). Its download needs internet access. If it fails, the real error is printed instead of being hidden.

In [4]:
englishExamples = ["Tokenization is unbelievably useful.", "I don't know."]

try:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

    for text in englishExamples:
        pieces = tokenizer.tokenize(text)
        print("Original text:", repr(text))
        print("Sub-word pieces:", pieces, "| Count:", len(pieces))
        print("Whitespace count:", len(whitespaceTokens(text)), "| Regex count:", len(regexTokens(text)))
        print()
except Exception as error:
    print("Model tokenizer NOT available:", type(error).__name__)
    print(str(error)[:200])
    print("Limitation recorded: sub-word results were not produced in this run.")

/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model tokenizer NOT available: OSError
Can't load the configuration of 'sentence-transformers/all-MiniLM-L6-v2'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same n
Limitation recorded: sub-word results were not produced in this run.


_The cell above prints the MiniLM sub-word pieces when the model can be downloaded. Where it prints the limitation message instead, no sub-word counts were measured, and none are claimed here. Re-run that cell with internet access and compare the counts with the two columns above._

## Part 2: FAQ Chatbot (TF-IDF)

The chatbot has eight FAQs, each with a source. It answers exact questions directly, asks for confirmation on similar questions, and says it has no answer when nothing is close enough.

Changes from the first version: the vectorizer is now fitted on the *cleaned* questions (it was fitted on raw text but queried with cleaned text, so `TF-IDF` and `tf-idf` could never match); `cleanText` now turns all punctuation into spaces; two FAQs were added so the "What can you do?" answer is true; and the code is runnable without typing (see the scripted cell).

In [5]:
# If scikit-learn is missing in this kernel, uncomment the next line, run it once, then restart the kernel.
# %pip install scikit-learn

In [6]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

faqs = [
    {
        "question": "Hello",
        "answer": "Hey! What can I do for you?",
        "source": "About this chatbot (written for this lab task)"
    },
    {
        "question": "Who are you?",
        "answer": "Hey! I'm a small FAQ chatbot made for a lab task. You ask me something and I try to find the best answer from my list.",
        "source": "About this chatbot (written for this lab task)"
    },
    {
        "question": "What can you do?",
        "answer": "I can answer a few basic questions, like what tokenization is, what TF-IDF is, and how I find my answers. If your question is a bit different from mine, I'll just check with you first.",
        "source": "About this chatbot (written for this lab task)"
    },
    {
        "question": "How do you find the answers?",
        "answer": "I compare the words in your question with the questions in my list using TF-IDF. Whichever one is closest to yours is the one I pick.",
        "source": "scikit-learn docs, Text feature extraction (tf-idf term weighting)"
    },
    {
        "question": "What is tokenization?",
        "answer": "Tokenization splits text into smaller units called tokens. Depending on the method, a token can be a space-separated string, a word-like piece, or a sub-word.",
        "source": "Lab 6 manual (CAI601400), section 6.1 Tokenization"
    },
    {
        "question": "What is TF-IDF?",
        "answer": "TF-IDF scores a word higher when it is frequent in one text but rare across all the texts, so distinctive words count more than common ones.",
        "source": "scikit-learn docs, Text feature extraction (tf-idf term weighting)"
    },
    {
        "question": "Where do we submit this?",
        "answer": "Create a Lab 6 folder in your GitHub repository and commit it there.",
        "source": "About this chatbot (written for this lab task)"
    },
    {
        "question": "Can you understand Urdu?",
        "answer": "Not really. My questions are in English, so I match English words best. Urdu words won't match anything in my list.",
        "source": "About this chatbot (written for this lab task)"
    }
]

# A suggestion is offered only when the best cosine similarity reaches this value.
# 0.3 is a hand-picked starting point for this tiny list, not a calibrated cutoff.
threshold = 0.3


def cleanText(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)      # punctuation -> space ("TF-IDF?" -> "tf idf ")
    return " ".join(text.split())              # collapse repeated spaces


questions = [faq["question"] for faq in faqs]

vectorizer = TfidfVectorizer()
faqMatrix = vectorizer.fit_transform([cleanText(q) for q in questions])


def findExactFaq(userText):
    for i in range(len(faqs)):
        if cleanText(faqs[i]["question"]) == cleanText(userText):
            return i
    return -1


def findBestFaq(userText):
    userVector = vectorizer.transform([cleanText(userText)])
    scores = cosine_similarity(userVector, faqMatrix)[0]

    bestIndex = 0
    for i in range(len(scores)):
        if scores[i] > scores[bestIndex]:
            bestIndex = i
    return bestIndex, scores[bestIndex]


def showAnswer(faq):
    print("Bot:", faq["answer"])
    print("Source:", faq["source"])


def chat():
    print("FAQ Chatbot. Type quit to stop.")

    while True:
        userText = input("You: ")

        if cleanText(userText) == "quit":
            print("Bot: Goodbye!")
            break

        if cleanText(userText) == "":
            print("Bot: Please type a question.")
            continue

        exactIndex = findExactFaq(userText)
        if exactIndex != -1:
            showAnswer(faqs[exactIndex])
            continue

        bestIndex, score = findBestFaq(userText)

        if score >= threshold:
            print("Bot: Did you mean:", faqs[bestIndex]["question"], "(yes/no)")
            reply = input("You: ")

            if cleanText(reply) == "yes":
                showAnswer(faqs[bestIndex])
            else:
                print("Bot: Okay, please try asking in a different way.")
        else:
            print("Bot: Sorry, I do not have an answer for that.")

### Run the chatbot (recorded transcript)

`chat()` waits for typing, which leaves no saved output when a notebook is run top to bottom. The cell below feeds the same `chat()` function a fixed list of inputs and echoes each one, so the transcript is captured in the notebook. It covers: an exact answer, a suggestion that the user accepts (`yes`), a suggestion that the user rejects (`no`), an empty line, an unsupported question, and `quit`. To try it by hand, run `chat()` in a new cell instead.

In [7]:
import builtins

scripted = [
    "What can you do?",          # exact answer
    "tell me who you are",       # similar -> suggestion
    "yes",                       # user confirms
    "what is tokenisation",      # near-miss spelling -> may or may not suggest
    "no",                        # user rejects (only used if a suggestion was offered)
    "",                          # empty input
    "Tell me a joke",            # unsupported
    "quit",
]
_queue = list(scripted)


def fakeInput(prompt=""):
    value = _queue.pop(0)
    print(prompt + value)
    return value


_realInput = builtins.input
builtins.input = fakeInput
try:
    chat()
finally:
    builtins.input = _realInput
    print("\n(unused scripted inputs:", _queue, ")")

FAQ Chatbot. Type quit to stop.
You: What can you do?
Bot: I can answer a few basic questions, like what tokenization is, what TF-IDF is, and how I find my answers. If your question is a bit different from mine, I'll just check with you first.
Source: About this chatbot (written for this lab task)
You: tell me who you are
Bot: Did you mean: Who are you? (yes/no)
You: yes
Bot: Hey! I'm a small FAQ chatbot made for a lab task. You ask me something and I try to find the best answer from my list.
Source: About this chatbot (written for this lab task)
You: what is tokenisation
Bot: Did you mean: What is tokenization? (yes/no)
You: no
Bot: Okay, please try asking in a different way.
You: 
Bot: Please type a question.
You: Tell me a joke
Bot: Sorry, I do not have an answer for that.
You: quit
Bot: Goodbye!

(unused scripted inputs: [] )


### Scores behind each decision

The similarity score is what decides between a suggestion and a refusal. The table shows the best match and its score for each non-exact query (threshold = 0.3).

In [8]:
for query in ["tell me who you are", "what is tokenisation", "Tell me a joke", "میں طالب علم ہوں۔", "how do you find answers"]:
    index, score = findBestFaq(query)
    decision = "suggest" if score >= threshold else "refuse"
    print(f"{query!r:34} best={faqs[index]['question']!r:32} score={score:.3f} -> {decision}")

'tell me who you are'              best='Who are you?'                   score=1.000 -> suggest
'what is tokenisation'             best='What is tokenization?'          score=0.742 -> suggest
'Tell me a joke'                   best='Hello'                          score=0.000 -> refuse
'میں طالب علم ہوں۔'                best='Hello'                          score=0.000 -> refuse
'how do you find answers'          best='How do you find the answers?'   score=0.893 -> suggest
